# Agentic Loop

Extends function calling into a full agent: instead of a fixed two-call exchange, we run a `while` loop that keeps executing until the model stops requesting tools. The model decides how many searches to make, what queries to use, and when it has enough information to answer.

In [1]:
import os
import json
from sqlitesearch import TextSearchIndex
from openai import OpenAI
from dotenv import load_dotenv
from zoomcamp.ingest import load_faq_data, build_index
from zoomcamp.rag_helper import RAGBase

In [2]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI(api_key=api_key)

In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

## System Instructions

The instructions explicitly tell the model to make *multiple* searches and to expand its queries based on what it finds. This is the key difference from simple function calling — we're not just giving it a tool, we're instructing it to use that tool iteratively.

In [4]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [5]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [6]:
index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="db/faq.db"
)

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

## First Turn — Model Makes Multiple Tool Calls

Send the first request. The model immediately issues two `search` calls in parallel (the OpenAI Responses API supports multiple tool calls in one turn). We collect them, dispatch each via `make_call`, and append results to `messages` — but we don't call the API again yet.

In [8]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration eligibility new student"}
function_call: search {"query":"course FAQ join course enrollment can I join after course has started"}
function_call: search {"query":"new student discovered the course can I still join enrollment access"}


In [9]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. If you’re just looking to learn, you can start anytime.

If you want, I can also help with what to do first after joining, or explain the certificate requirements.


## The Agent Loop

The `while True` loop drives the agent: call the API, dispatch any tool calls, append results, repeat. When the model returns a message with no tool calls (`has_function_calls == False`), it's done — break out and return the final answer.

In [10]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [11]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run Ollama install local server FAQ"}
function_call: search {"query":"Ollama local run model command FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)
   - **Windows**: download the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and start a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **Use it from Python**
   ```bash
   pip install ollama
   ```
   Example:
   ```python
   import ollama

   response = ollama.chat(
       model='llama3',
       messages=[{"role": "user", "content": "Hello!"}]
   )

   print(response['message'][

'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - **macOS**: download the `.pkg` from [ollama.com/download](https://ollama.com/download)\n   - **Windows**: download the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and start a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **Use it from Python**\n   ```bash\n   pip install ollama\n   ```\n   Example:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you get a **connection refused** error, restart the server with:\n```bash\nollama serve\n```\nor in a notebook:\n```bash\n!nohup

## Refining the Instructions — Off-Topic Guardrail

Without explicit constraints, the model answers off-topic questions from its own knowledge (e.g. chess) even when the FAQ has nothing relevant. Adding "only use facts from the FAQ database" forces the model to stay grounded and refuse questions it can't answer from the index.

In [12]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course still join enrollment late registration"}
function_call: search {"query":"can I still join the course after it started enrollment late join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, registration isn’t required to start learning; you can begin the materials and submit homework while the forms are open.

If you’d like, I can also help with what you should do first after joining.


'Yes — you can still join the course.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, registration isn’t required to start learning; you can begin the materials and submit homework while the forms are open.\n\nIf you’d like, I can also help with what you should do first after joining.'

In [13]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enroll discovered course can I join late registration FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while submissions open homework form open not need registration FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key requirement is to submit your project while submissions are still open. You can also start learning and submitting homework even if you didn’t register earlier.

If you’d like, I can also help with what to do next after joining.


'Yes — you can still join the course.\n\nIf you want a certificate, the key requirement is to submit your project while submissions are still open. You can also start learning and submitting homework even if you didn’t register earlier.\n\nIf you’d like, I can also help with what to do next after joining.'

In [14]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening queen pawn d4 d5"}
iteration #3...
ASSISTANT:
The **Queen’s Gambit** is a classic **chess opening**.

It starts with:

1. **d4 d5**
2. **c4**

White offers the **c-pawn** as a pawn sacrifice to try to control the center and gain a positional advantage.  
Despite the name, it’s **not really a gambit** in the usual sense, because Black can often hold the pawn and White usually gets strong compensation in space and development.

Common main branches include:

- **Queen’s Gambit Declined**: Black does not take the pawn
- **Queen’s Gambit Accepted**: Black takes the pawn on c4

If you want, I can also explain **why it’s called a gambit**, or show a **simple board diagram / move ideas**.


'The **Queen’s Gambit** is a classic **chess opening**.\n\nIt starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the **c-pawn** as a pawn sacrifice to try to control the center and gain a positional advantage.  \nDespite the name, it’s **not really a gambit** in the usual sense, because Black can often hold the pawn and White usually gets strong compensation in space and development.\n\nCommon main branches include:\n\n- **Queen’s Gambit Declined**: Black does not take the pawn\n- **Queen’s Gambit Accepted**: Black takes the pawn on c4\n\nIf you want, I can also explain **why it’s called a gambit**, or show a **simple board diagram / move ideas**.'

In [15]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
ASSISTANT:
I couldn’t find anything in the course FAQ about **queen’s gambit**, so I think this is off-topic for the course.

If you meant the chess opening: the **Queen’s Gambit** is a chess opening that starts with:
1. d4 d5  
2. c4

It’s a very common opening where White offers a pawn to try to gain control of the center.

If you want, I can help with other course-related questions too—anything else you want to explore?


'I couldn’t find anything in the course FAQ about **queen’s gambit**, so I think this is off-topic for the course.\n\nIf you meant the chess opening: the **Queen’s Gambit** is a chess opening that starts with:\n1. d4 d5  \n2. c4\n\nIt’s a very common opening where White offers a pawn to try to gain control of the center.\n\nIf you want, I can help with other course-related questions too—anything else you want to explore?'